In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import re
from datasets import load_dataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
#from cleantext import clean
#import pytorch_lightning as pl
from datasets import Dataset as dts
#from sentence_transformers import SentenceTransformer
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn.utils.rnn import pad_sequence
from torchvision.ops import focal_loss
from torch.utils.data import WeightedRandomSampler

In [2]:
import logging
from pathlib import Path


In [3]:
!pip install mawo-razdel emoji

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip


In [4]:
import numpy as np
import emoji

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
dataset = load_dataset('astromis/presuicidal_signals')


In [7]:
ds_train = dataset['train'].to_pandas()
ds_test = dataset['test'].to_pandas()

In [8]:
def emoji_to_clean_russian(text):
    clean_tags = re.sub(r'<emoji>\s*(.*?)\s*</emoji>', lambda m: f":{m.group(1).strip().replace(' ', '_').lower()}:", text)
    emojified = emoji.emojize(clean_tags, language='en')
    

    ru_demojized = emoji.demojize(emojified, language='ru')
    

    final_text = ru_demojized.replace('::', ' ').replace(':', ' ').replace('_', ' ')
    final_text = re.sub(r'\s+', ' ', final_text).strip()
    final_text = re.sub(r'[,;:«»""()\[\]{}]', ' ', final_text)
    return final_text

In [9]:
ds_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 22787 entries, 0 to 22786
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    22787 non-null  str  
 1   label   22787 non-null  int64
dtypes: int64(1), str(1)
memory usage: 3.9 MB


In [10]:
print(f'MEAN LABEL ON TRAIN: {ds_train['label'].mean()}')
print(f'MEAN LABEL ON TEST: {ds_test['label'].mean()}')

MEAN LABEL ON TRAIN: 0.15017334445078334
MEAN LABEL ON TEST: 0.15542131667861164


In [11]:
print(ds_train.isna().sum())
print(ds_test.isna().sum())

text     0
label    0
dtype: int64
text     0
label    0
dtype: int64


In [12]:
def cleaning_text(text):
    text = re.sub(r"</?emoji>", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [13]:
ds_test['text'] = ds_test['text'].apply(emoji_to_clean_russian)
ds_train['text'] = ds_train['text'].apply(emoji_to_clean_russian)

In [14]:
ds_train['text_len'] = ds_train['text'].str.len()
ds_test['text_len'] = ds_test['text'].str.len()

In [15]:
print(ds_train['text_len'].mean())
print(ds_train['text_len'].min())
print(ds_train['text_len'].max())

90.8253828937552
0
2582


In [16]:
ds_train['count_words'] = ds_train['text'].str.split().str.len()
ds_test['count_words'] = ds_test['text'].str.split().str.len()

In [17]:
print(ds_train['count_words'].mean())
print(ds_train['count_words'].min())
print(ds_train['count_words'].max())

15.44753587571861
0
421


In [18]:
print(len(ds_train[ds_train['count_words'] == 2]))
print(len(ds_test[ds_test['count_words'] == 0]))

923
9


In [19]:
ds_train[ds_train['count_words'] ==2]

,text,label,text_len,count_words
6,слава кпсс!!,0,12,2
12,Давайте поболтаем,0,17,2
18,Есть желающие?#закос,0,20,2
21,люблю герлловвввер,0,18,2
41,Я ЕБАОА,0,7,2
...,...,...,...,...
22592,изгибы тельцА,0,13,2
22596,паучье молоко,0,13,2
22672,я живая,0,7,2
22711,хочу каре...,0,12,2


In [20]:
ds_train['text'][333]

'бог подарил мне подарок в честь дня рождения стэй в лице человека который не смог пить пакетированный чай и сам взял заварил вкуснейший чай бля слезы рекой слезы рекой чел религия религия'

In [21]:
ds_train = ds_train[ds_train['count_words'] >=2]
len(ds_train)

22092

In [22]:
y_test = ds_test['label']
X_test = ds_test.drop(columns=['label'])

In [23]:
y = ds_train['label']
X = ds_train.drop(columns=['label'])


In [24]:
X_train, X_valid, y_train, y_valid = train_test_split(X,  y, test_size=0.2, stratify=y, random_state=42)

In [25]:
X_train.head()


,text,text_len,count_words
3485,я прочел все по дошкольному возрасту страшно в...,86,15
20342,приснилось что я трахаюсь с женщиной,36,6
4024,Так всё хорош на ближайшую неделю мне хвати...,93,16
19444,Только что заметил что там диор. И Я ДЕЛАЮ ВД...,96,19
1323,Никогда не смогу доверять человеку но так хоч...,71,10


In [26]:
from mawo_razdel import tokenize
from collections import Counter

In [27]:
from collections import Counter
import numpy as np


class RNNTokenizer:
    
    def __init__(self, tokenize_fn=None, min_freq=2, max_len=None, lowercase=True):
        self.tokenize_fn = tokenize_fn or (lambda s: s.split())
        self.min_freq = min_freq
        self.max_len = max_len
        self.lowercase = lowercase

        self.vocab = {'<PAD>': 0, '<UNK>': 1}
        self.pad_token_id = 0
        self.unk_token_id = 1

    def fit(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(self._tokenize(text))
        for token, freq in counter.most_common():
            if freq >= self.min_freq:
                self.vocab[token] = len(self.vocab)
        return self

    def fit_max_len(self, texts, pct=99):
        lengths = np.array([len(self._tokenize(t)) for t in texts])
        self.max_len = int(np.percentile(lengths, pct))
        return self.max_len, lengths.mean(), np.percentile(lengths, pct)

    def encode(self, text, **kwargs):
        output = {'input_ids': []}
        for t in text:
            tokens = self._tokenize(t)
            ids = [self.vocab.get(t, self.unk_token_id) for t in tokens]
            if self.max_len is not None:
                ids = ids[:self.max_len]
            output['input_ids'].append(ids)   
        return output

    def __call__(self, text):
        return self.encode(text)

    def _tokenize(self, text):
        tokens = self.tokenize_fn(text)
        return [t.text.lower() for t in tokens] if self.lowercase else list(tokens)

    def __len__(self):
        return len(self.vocab)

In [28]:
tokenizer_rnn = RNNTokenizer(tokenize_fn=tokenize)
tokenizer_rnn.fit_max_len(X_train['text'])
tokenizer_rnn.fit(X_train['text'])
vocab_size = len(tokenizer_rnn.vocab)

In [29]:
class MyDataset(Dataset):

    

    def __init__(self, input, tokenizer, labels, attention_mask=True):

        tokenized_text = tokenizer(input.tolist(), padding=False, truncation=True, max_length=128, return_tensors=None)

        self.enc = tokenized_text

        self.labels = labels.tolist()
        
                

    def __len__(self):
        return len(self.labels)


    def __getitem__(self, index):
        item = {k: torch.tensor(v[index], dtype=torch.long) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


In [30]:
def collate_fn(batch, pad_value):
    padding_value = pad_value

    sequences = [item["ids"] for item in batch]
    labels    = [item["labels"]    for item in batch]


    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=padding_value)
    labels = torch.stack(labels)
    return {
        'ids' : padded_sequences, 'labels': labels
    }
    


In [31]:
train_dataset = MyDataset(input=X_train['text'], labels=y_train, tokenizer=tokenizer_rnn.encode, attention_mask=False)
val_dataset = MyDataset(input=X_valid['text'], labels=y_valid, tokenizer=tokenizer_rnn.encode, attention_mask=False)
train_dataloader = DataLoader(train_dataset, collate_fn=collate_fn, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, collate_fn=collate_fn, batch_size=4, shuffle=True)

In [32]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_index=0):
        super().__init__()

        self.embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            padding_idx=0,
            embedding_dim=embed_dim
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if num_layers >1 else 0.0
        )
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(hidden_dim* 2, 1)
        self.pad_index = pad_index

    def forward(self, input_ids):

        lengths = (input_ids != self.pad_index).sum(dim=-1)
        embeds = self.embeddings(input_ids) # -> [batch_size, seq_len, embed_dim]

        packed = pack_padded_sequence(embeds, lengths.cpu(), batch_first=True, enforce_sorted=False)


        _, (h_n, _) = self.lstm(packed)

        h = torch.cat([h_n[-2], h_n[-1]],dim=-1)

        logits = self.fc(self.dropout(h))

        return logits.squeeze(1)




In [33]:
from sklearn.metrics import recall_score, auc, f1_score, precision_recall_curve
from sklearn.metrics import roc_auc_score
import gc


gc.collect()

0

После нескольких тестовых запусков, видно, что после 1 эпохи метрики начинают падать, так же, как и  вероятности для положительного класса, зная, что целевого класса 15%, можно попробовать сделать оверсемплинг.

In [34]:
class_counts = np.bincount(y_train)
class_weight = 1.0/ class_counts
sample_weights = class_weight[y_train]
sampler = WeightedRandomSampler(
weights=torch.from_numpy(sample_weights).double(), num_samples=len(sample_weights),
replacement=True
)
pos_weight = torch.tensor([((1 - y_train.mean()) / y_train.mean())], device=device)

Лучший roc-auc был 0.64, попробуем использовать энкодер модель.

In [35]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, accuracy_score
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

In [36]:
model_rnn = RNNClassifier(vocab_size=vocab_size, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.3).to(device)

In [ ]:
model_id = "deepvk/deberta-v1-distill"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, torch_dtype=torch.float32, num_labels=1, ignore_mismatched_sizes=True, problem_type='multi_label_classification')
pad_token_id = tokenizer.pad_token_id
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/669 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/332 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/455M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: deepvk/bert-base-uncased
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [38]:
encoder_test_ds = MyDataset(X_test['text'], labels=y_test, tokenizer=tokenizer)
encoder_test_loader = DataLoader(encoder_test_ds, collate_fn=data_collator, batch_size=4)

In [39]:
encoder_train_ds = MyDataset(X_train['text'], labels=y_train, tokenizer=tokenizer)
encoder_valid_ds = MyDataset(X_valid['text'], labels=y_valid, tokenizer=tokenizer)
encoder_train_loader = DataLoader(encoder_train_ds, collate_fn=data_collator, batch_size=16) #sampler=sampler)
encoder_valid_loader = DataLoader(encoder_valid_ds, collate_fn=data_collator, batch_size=16)

In [40]:
no_decay = ["bias", "norm", "rmsnorm", "layernorm"]

optimizer_grouped_parameters = [
    {
        "params": [
            p for name, p in model.named_parameters()
            if p.requires_grad
            and not any(nd in name.lower() for nd in no_decay)
        ],
        "weight_decay": 0.01,
    },
    {
        "params": [
            p for name, p in model.named_parameters()
            if p.requires_grad
            and any(nd in name.lower() for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

In [41]:
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=2e-5, eps=1e-8, weight_decay=0.01, betas=(0.9, 0.95))
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.5, end_factor=1.0, total_iters=len(encoder_train_loader))

In [42]:
try:
    next(iter(encoder_train_loader))
except Exception:
    'Ошибка лоадера'

In [43]:
class ModelAdapter:

    def forward(self, model, batch):
        raise NotImplementedError

    

In [44]:
class HFModelAdapter(ModelAdapter):

    def forward(self, model, batch):

        output = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

        logits = output.logits.float().squeeze(1)

        return logits

        

In [45]:
class RNNAdapter(ModelAdapter):

    def forward(self, model, batch):

        logits = model(batch['input_ids'])

        return logits
        



In [46]:
import copy
class MyTrainer:
    def __init__(
        self, model, name, log_file,
        criterion, train_loader, adapter: ModelAdapter,
        valid_loader, test_loader, optimizer,
        scheduler=None, device=None,
        early_stopping=None, thresholds=None,
    ):
        Path(log_file).parent.mkdir(parents=True, exist_ok=True)

        self.logger = logging.getLogger(f"Trainer.{Path(log_file).resolve()}")
        self.logger.setLevel(logging.INFO)
        self.logger.propagate = False

        file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
        file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))

        self.logger.addHandler(file_handler)

        self.adapter = adapter
        self.model = model
        self.name = name
        self.criterion = criterion
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.test_loader = test_loader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.early_stopping = early_stopping

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        self.thresholds = (thresholds if thresholds is not None else np.arange(0.01, 1.00, 0.001))

        self.history = []

        self.best_state = None
        self.best_valid_loss = float("inf")

    def train_epoch(self):
        self.model.train()
        train_loss = 0.0

        progress_bar = tqdm(self.train_loader, desc="Training")

        for batch in progress_bar:
            batch = {k: v.to(self.device) for k, v in batch.items()}

            labels = batch["labels"].float()

            self.optimizer.zero_grad(set_to_none=True)

            logits = self.adapter.forward(self.model, batch) 

            assert logits.shape 

            assert torch.isfinite(logits).all(), 'Nan logits'

            loss = self.criterion(logits, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step()

            if self.scheduler is not None:
                self.scheduler.step()

            train_loss += loss.item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

        return train_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self, plot_pr=True):
        self.model.eval()

        valid_loss = 0.0
        all_probs = []
        all_labels = []

        for batch in tqdm(self.valid_loader, desc="Validation"):
            batch = {k: v.to(self.device) for k, v in batch.items()}

            labels = batch["labels"].float()

            logits = self.adapter.forward(self.model, batch)

            assert torch.isfinite(logits).all(), 'Nan logits'


            loss = self.criterion(logits, labels)
            valid_loss += loss.item()

            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

        valid_loss/=len(self.valid_loader)

        all_probs = torch.cat(all_probs).numpy()
        all_labels = torch.cat(all_labels).numpy()

        roc_auc = roc_auc_score(all_labels, all_probs)

        ap = average_precision_score(all_labels, all_probs)
        precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)

        if plot_pr:
            

            plt.figure(figsize=(6, 5))
            plt.plot(recall, precision, label=f"AP = {ap:.3f}")
            plt.axhline(y=all_labels.mean(), color="gray", linestyle="--", label=f"Baseline = {all_labels.mean():.3f}")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.title("Precision-Recall curve")
            plt.legend()
            plt.grid(True)
            plt.show()

        metrics = []

        for threshold in self.thresholds:
            preds = (all_probs >= threshold).astype(int)

            precision = precision_score(all_labels, preds, zero_division=0)
            recall = recall_score(all_labels, preds, zero_division=0)
            f1 = f1_score(all_labels, preds, zero_division=0)
            acc = accuracy_score(all_labels, preds)


            self.logger.info(
                f"Name: {self.name} | "
                f"Threshold: {threshold:.3f} | "
                f"Precision: {precision:.3f} | "
                f"Recall: {recall:.3f} | "
                f"F1: {f1:.3f}"
)
        
        return {
            "valid_loss": valid_loss,
            "roc_auc": roc_auc,
            "average_precision": ap,
            "metrics": metrics,
        }


    @torch.no_grad()
    def run_test(self, threshold):


        all_preds = []
        all_labels = []
        for batch in tqdm(self.test_loader, desc="Testing"):
            batch = {k: v.to(self.device) for k, v in batch.items()}

            labels = batch["labels"].float()
            
            logits = self.adapter.forward(self.model, batch)

            assert torch.isfinite(logits).all(), 'Nan logits'

            probs = torch.sigmoid(logits)

            preds = (probs >= threshold).long()
            
            all_preds.append(preds.cpu())

            all_labels.append(labels.cpu())

        all_labels = torch.cat(all_labels).numpy()
        
        all_preds = torch.cat(all_preds).numpy()

        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        acc = accuracy_score(all_labels, all_preds)

        self.logger.critical(f'TEST RESULT: precision: {precision:.3f} | recall: {recall:.4f} | f1: {f1:.3f} | accuracy: {acc:.3f}')


        return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": acc,
        }

    

    def fit(self, epochs,
        plot_pr=True, print_all_thresholds=True,
        restore_best=True, patience=2
    ):
        for epoch in range(epochs):
            print(f"\nEpoch {epoch + 1}/{epochs}")

            train_loss = self.train_epoch()
            val = self.validate(plot_pr=plot_pr)
            self.logger.warning(f'Name: {self.name} | Validation loss: {val['valid_loss']:.3f} | Validation roc-auc: {val['roc_auc']:.3f}')

            valid_loss = val['valid_loss']
            
            if valid_loss < self.best_valid_loss:
                self.best_valid_loss = valid_loss
                self.best_state = copy.deepcopy(
                    self.model.state_dict()
                )
                patience_counter=0
            else:
                patience_counter+=1

            if patience_counter >= patience:
                print(f'Early stop triggered after {epoch} epoch')
                break

            self.history.append({'validate_metrics': val, 'train_loss': train_loss})
            
            

        if restore_best and self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            print("Restored best model by validation loss.")
            print(f'Best validation loss: {self.best_valid_loss:.3f}')


        return self.history


In [47]:
trainer = MyTrainer(model=model, name='modelL', log_file='model.log',criterion=torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight), scheduler=scheduler,
                    valid_loader=encoder_valid_loader, train_loader=encoder_train_loader, adapter=HFModelAdapter(), test_loader=encoder_test_loader, optimizer=optimizer, device=device)

In [48]:
trainer.fit(epochs=5)


Epoch 1/5


Training:   0%|          | 0/1105 [00:00<?, ?it/s]

/home/ydlawq/torch_rocm_v/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:338.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


KeyboardInterrupt: 

In [103]:
trainer.run_test(0.152)

Testing:   0%|          | 0/2442 [00:00<?, ?it/s]

{'threshold': 0.152,
 'precision': 0.33680639689847347,
 'recall': 0.9156785243741765,
 'f1': 0.4924712134632418,
 'accuracy': 0.7066653015255452}